# Bilayer stackings

## Extraction of gray values in the same location

In [2]:
import csv
import random
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
from PIL import Image


# =============================================================================
# 1. Path and parameter settings
# =============================================================================

DATABASE_ROOT = Path(r"G:\DiffStack-code\Symbolic-regression\computem4sym")
OUTPUT_ROOT = Path(r"G:\DiffStack-code\Symbolic-regression\gray_results\computem4sym")

IMAGE_EXTENSIONS = (".png", ".tif", ".tiff")
TARGET_PIXEL_COUNT = 400
RANDOM_SEED = 21

# Your current structure is:
# DATABASE_ROOT/layer1/data
# DATABASE_ROOT/layer2/data
# DATABASE_ROOT/bilayer/data
USE_NESTED_LAYER_DIR = True

PARAM_PATTERN = re.compile(
    r"^([-+]?\d+(?:\.\d+)?)_"
    r"([-+]?\d+(?:\.\d+)?)_"
    r"([-+]?\d+(?:\.\d+)?)_"
    r"([-+]?\d+(?:\.\d+)?)_"
)

TARGET_STACK_PARAMS_LIST = [
    "-3.2_-2.8_0_0",
]


# =============================================================================
# 2. Basic utilities
# =============================================================================

def init_output_dir() -> None:
    """Create the output directory."""
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Output directory: {OUTPUT_ROOT}")


def raw_params_to_full_params(raw_params: str) -> str:
    """
    Convert raw parameter format into the unified index format.

    Example
    -------
    "0_1.8_0_0" -> "x0_y1.8_rot0_flip0"
    """
    parts = raw_params.split("_")

    if len(parts) != 4:
        raise ValueError(f"Expected format: x_y_rot_flip, but got: {raw_params}")

    x, y, rot, flip = parts
    return f"x{x}_y{y}_rot{rot}_flip{flip}"


def extract_stack_params(filename: str) -> Optional[str]:
    """
    Extract stacking parameters from an image filename.

    Returns
    -------
    str or None
        Unified parameter index:
        x{x}_y{y}_rot{rot}_flip{flip}
    """
    match = PARAM_PATTERN.match(filename)

    if match is None:
        return None

    x, y, rot, flip = match.group(1), match.group(2), match.group(3), match.group(4)
    return f"x{x}_y{y}_rot{rot}_flip{flip}"


def get_layer_data_dir(layer_type: str) -> Path:
    """
    Return the image directory for a given layer.

    Supported directory layouts
    ---------------------------
    1. DATABASE_ROOT/data_layer1
       DATABASE_ROOT/data_layer2
       DATABASE_ROOT/data_bilayer

    2. DATABASE_ROOT/layer1/data
       DATABASE_ROOT/layer2/data
       DATABASE_ROOT/bilayer/data
    """
    if USE_NESTED_LAYER_DIR:
        return DATABASE_ROOT / layer_type / "data"

    return DATABASE_ROOT / f"data_{layer_type}"


def get_image_paths_by_layer(layer_type: str) -> Dict[str, Path]:
    """
    Build an image-path dictionary for a given layer.

    Returns
    -------
    dict
        Key   : unified stacking parameter index
        Value : image path
    """
    layer_dir = get_layer_data_dir(layer_type)
    image_paths = {}

    if not layer_dir.exists():
        print(f"Error: layer directory does not exist: {layer_dir}")
        return image_paths

    for file_path in layer_dir.iterdir():
        if not file_path.is_file():
            continue

        if file_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        full_params = extract_stack_params(file_path.name)

        if full_params is None:
            print(f"Skipped file with unrecognized parameter format: {file_path.name}")
            continue

        image_paths[full_params] = file_path

    print(f"{layer_type}: loaded {len(image_paths)} images")
    return image_paths


def load_image_as_gray_array(
    image_path: Path,
    target_size: Tuple[int, int] = (1024, 1024),
) -> Optional[np.ndarray]:
    """
    Load an image as a grayscale NumPy array.

    If the image size is not target_size, a center crop is applied.
    """
    try:
        with Image.open(image_path).convert("L") as img:
            if img.size != target_size:
                left = img.size[0] // 2 - target_size[0] // 2
                top = img.size[1] // 2 - target_size[1] // 2
                img = img.crop((left, top, left + target_size[0], top + target_size[1]))

            return np.asarray(img, dtype=np.float32)

    except Exception as exc:
        print(f"Failed to load image: {image_path.name}. Reason: {exc}")
        return None


# =============================================================================
# 3. Valid-pixel selection
# =============================================================================

def mean_filter_5x5(image_arr: np.ndarray) -> np.ndarray:
    """
    Compute the 5 x 5 local mean gray value for each pixel.

    Edge pixels are handled by edge padding.
    """
    height, width = image_arr.shape
    padded = np.pad(image_arr, pad_width=2, mode="edge")

    result = np.zeros_like(image_arr, dtype=np.float32)

    for dy in range(5):
        for dx in range(5):
            result += padded[dy:dy + height, dx:dx + width]

    return result / 25.0


def get_valid_pixels(
    layer1_arr: np.ndarray,
    layer2_arr: np.ndarray,
    bilayer_arr: np.ndarray,
) -> List[Tuple[int, int, float, float, float]]:
    """
    Select valid pixels for absolute gray-value relationship fitting.

    Current criteria
    ----------------
    1. Bilayer center pixel > 50.
    2. Bilayer 5 x 5 local mean > 50.
    3. Layer1 or layer2 5 x 5 local mean > 30.

    Returns
    -------
    list
        Each item is:
        (x, y, layer1_gray, layer2_gray, bilayer_gray)

        The gray values are absolute 5 x 5 local means, not centered or normalized.
    """
    l1_avg = mean_filter_5x5(layer1_arr)
    l2_avg = mean_filter_5x5(layer2_arr)
    bil_avg = mean_filter_5x5(bilayer_arr)

    mask = (
        (bilayer_arr > 50)
        & (bil_avg > 50)
        & ((l1_avg > 30) | (l2_avg > 30))
    )

    ys, xs = np.where(mask)

    return [
        (
            int(x),
            int(y),
            float(l1_avg[y, x]),
            float(l2_avg[y, x]),
            float(bil_avg[y, x]),
        )
        for y, x in zip(ys, xs)
    ]


# =============================================================================
# 4. CSV writing and statistics
# =============================================================================

def compute_gray_stats(values: List[float]) -> Dict[str, float]:
    """Compute basic gray-value statistics."""
    arr = np.asarray(values, dtype=np.float32)

    mean = float(np.mean(arr))
    std = float(np.std(arr))
    cv = float(std / mean) if mean != 0 else 0.0
    q25, q75 = np.percentile(arr, [25, 75])

    return {
        "mean": mean,
        "std": std,
        "cv": cv,
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "q25": float(q25),
        "q75": float(q75),
    }


def write_pixel_csv(
    output_csv_path: Path,
    selected_pixels: List[Tuple[int, int, float, float, float]],
) -> None:
    """Save selected absolute gray values to CSV."""
    output_csv_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["x", "y", "layer1_gray", "layer2_gray", "bilayer_gray"])
        writer.writerows(selected_pixels)


def append_summary_csv(
    summary_csv_path: Path,
    params_tag: str,
    selected_pixels: List[Tuple[int, int, float, float, float]],
) -> None:
    """Append gray-value statistics for one stacking configuration."""
    summary_csv_path.parent.mkdir(parents=True, exist_ok=True)

    l1_stats = compute_gray_stats([p[2] for p in selected_pixels])
    l2_stats = compute_gray_stats([p[3] for p in selected_pixels])
    bil_stats = compute_gray_stats([p[4] for p in selected_pixels])

    file_exists = summary_csv_path.exists()

    header = [
        "params", "count",
        "l1_mean", "l1_std", "l1_cv", "l1_min", "l1_max", "l1_q25", "l1_q75",
        "l2_mean", "l2_std", "l2_cv", "l2_min", "l2_max", "l2_q25", "l2_q75",
        "bil_mean", "bil_std", "bil_cv", "bil_min", "bil_max", "bil_q25", "bil_q75",
    ]

    row = [
        params_tag, len(selected_pixels),
        l1_stats["mean"], l1_stats["std"], l1_stats["cv"], l1_stats["min"], l1_stats["max"], l1_stats["q25"], l1_stats["q75"],
        l2_stats["mean"], l2_stats["std"], l2_stats["cv"], l2_stats["min"], l2_stats["max"], l2_stats["q25"], l2_stats["q75"],
        bil_stats["mean"], bil_stats["std"], bil_stats["cv"], bil_stats["min"], bil_stats["max"], bil_stats["q25"], bil_stats["q75"],
    ]

    with open(summary_csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        if not file_exists:
            writer.writerow(header)

        writer.writerow(row)


# =============================================================================
# 5. Single-stack extraction
# =============================================================================

def extract_gray_values_for_stack(
    layer1_img_path: Path,
    layer2_img_path: Path,
    bilayer_img_path: Path,
    output_csv_path: Path,
    summary_csv_path: Optional[Path] = None,
) -> bool:
    """
    Extract absolute gray values from one layer1/layer2/bilayer image triplet.

    No centering or normalization is applied.
    """
    layer1_arr = load_image_as_gray_array(layer1_img_path)
    layer2_arr = load_image_as_gray_array(layer2_img_path)
    bilayer_arr = load_image_as_gray_array(bilayer_img_path)

    if any(arr is None for arr in [layer1_arr, layer2_arr, bilayer_arr]):
        print("Skipped: at least one image failed to load")
        return False

    if not (layer1_arr.shape == layer2_arr.shape == bilayer_arr.shape):
        print(
            "Skipped: image sizes are inconsistent "
            f"layer1={layer1_arr.shape}, layer2={layer2_arr.shape}, bilayer={bilayer_arr.shape}"
        )
        return False

    valid_pixels = get_valid_pixels(layer1_arr, layer2_arr, bilayer_arr)

    if not valid_pixels:
        print("Skipped: no valid pixels found")
        return False

    selected_pixels = (
        random.sample(valid_pixels, TARGET_PIXEL_COUNT)
        if len(valid_pixels) > TARGET_PIXEL_COUNT
        else valid_pixels
    )

    write_pixel_csv(output_csv_path, selected_pixels)

    if summary_csv_path is not None:
        params_tag = output_csv_path.stem.replace("gray_values_", "")
        append_summary_csv(summary_csv_path, params_tag, selected_pixels)

    print(f"Saved: {output_csv_path.name}, valid pixels: {len(selected_pixels)}")
    return True


# =============================================================================
# 6. Batch extraction
# =============================================================================

def batch_extract_gray_values() -> None:
    """
    Batch extract absolute gray values for TARGET_STACK_PARAMS_LIST.
    """
    init_output_dir()

    layer1_paths = get_image_paths_by_layer("layer1")
    layer2_paths = get_image_paths_by_layer("layer2")
    bilayer_paths = get_image_paths_by_layer("bilayer")

    summary_csv_path = OUTPUT_ROOT / "summary_gray_stats.csv"

    success_count = 0

    for raw_params in TARGET_STACK_PARAMS_LIST:
        try:
            full_params = raw_params_to_full_params(raw_params)
        except ValueError as exc:
            print(f"Invalid parameter: {exc}")
            continue

        print(f"\nProcessing: {raw_params} -> {full_params}")

        missing_layers = [
            layer_name
            for layer_name, paths in [
                ("layer1", layer1_paths),
                ("layer2", layer2_paths),
                ("bilayer", bilayer_paths),
            ]
            if full_params not in paths
        ]

        if missing_layers:
            print(f"Skipped: missing {missing_layers}")
            continue

        output_csv_path = OUTPUT_ROOT / f"gray_values_{full_params}.csv"

        ok = extract_gray_values_for_stack(
            layer1_paths[full_params],
            layer2_paths[full_params],
            bilayer_paths[full_params],
            output_csv_path,
            summary_csv_path,
        )

        if ok:
            success_count += 1

    total = len(TARGET_STACK_PARAMS_LIST)

    print("\nBatch extraction finished")
    print(f"Requested groups: {total}")
    print(f"Successful groups: {success_count}")
    print(f"Failed or missing groups: {total - success_count}")
    print(f"Result directory: {OUTPUT_ROOT}")


# =============================================================================
# 7. Main entry
# =============================================================================

if __name__ == "__main__":
    random.seed(RANDOM_SEED)
    batch_extract_gray_values()

Output directory: G:\DiffStack-code\Symbolic-regression\gray_results\computem4sym
layer1: loaded 4 images
layer2: loaded 4 images
bilayer: loaded 4 images

Processing: -3.2_-2.8_0_0 -> x-3.2_y-2.8_rot0_flip0
Saved: gray_values_x-3.2_y-2.8_rot0_flip0.csv, valid pixels: 400

Batch extraction finished
Requested groups: 1
Successful groups: 1
Failed or missing groups: 0
Result directory: G:\DiffStack-code\Symbolic-regression\gray_results\computem4sym


In [1]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pysr import PySRRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")


# =============================================================================
# 1. Path and training settings
# =============================================================================

GRAY_DATA_DIR = Path(r"G:\DiffStack-code\Symbolic-regression\gray_results\computem4sym")
RESULT_DIR = Path(r"G:\DiffStack-code\Symbolic-regression\regression_results\computem4sym_absolute")

POPULATION_SIZE = 100
GENERATIONS = 10
PARSIMONY_COEFFICIENT = 0.5
RANDOM_STATE = 42

RESULT_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 2. Data loading
# =============================================================================

def load_and_merge_gray_data(data_dir: Path) -> pd.DataFrame:
    """
    Load all gray-value CSV files and merge them into one DataFrame.

    Expected CSV columns
    --------------------
    layer1_gray
    layer2_gray
    bilayer_gray

    Additional metadata columns are added:
    source_image
    pixel_index
    """
    all_data = []

    for csv_path in sorted(data_dir.glob("gray_values_*.csv")):
        df = pd.read_csv(csv_path)

        required_cols = ["layer1_gray", "layer2_gray", "bilayer_gray"]
        missing_cols = [col for col in required_cols if col not in df.columns]

        if missing_cols:
            print(f"Skipped {csv_path.name}: missing columns {missing_cols}")
            continue

        df = df[required_cols].copy()
        df["source_image"] = csv_path.name
        df["pixel_index"] = df.index

        all_data.append(df)
        print(f"Loaded: {csv_path.name} | samples: {len(df)}")

    if not all_data:
        raise ValueError(f"No valid gray-value CSV files found in: {data_dir}")

    merged_df = pd.concat(all_data, ignore_index=True)
    print(f"\nMerged dataset size: {len(merged_df)} samples")

    return merged_df


# =============================================================================
# 3. Evaluation metrics
# =============================================================================

def mean_absolute_percentage_error(y_true, y_pred) -> float:
    """
    Compute MAPE while ignoring zero-valued targets.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mask = y_true != 0

    if not np.any(mask):
        return np.nan

    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def evaluate_prediction(y_true, y_pred) -> dict:
    """
    Compute common regression metrics.
    """
    return {
        "r2": r2_score(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mae": mean_absolute_error(y_true, y_pred),
        "mape": mean_absolute_percentage_error(y_true, y_pred),
        "mse": mean_squared_error(y_true, y_pred),
    }


# =============================================================================
# 4. Symbolic regression
# =============================================================================

def train_symbolic_regressor(X_train, X_test, y_train, y_test):
    """
    Train a PySR symbolic regression model.

    Model target
    ------------
    bilayer_gray = f(layer1_gray, layer2_gray)

    The input gray values are absolute gray values.
    No centering or normalization is applied.
    """
    print(f"\nTraining samples: {len(X_train)} | Test samples: {len(X_test)}")

    model = PySRRegressor(
        niterations=GENERATIONS,
        populations=POPULATION_SIZE,
        binary_operators=["+", "-", "*", "/", "pow"],
        unary_operators=[],
        maxsize=10,
        parsimony=PARSIMONY_COEFFICIENT,
        elementwise_loss="L2DistLoss()",
        model_selection="best",
        random_state=RANDOM_STATE,
        verbosity=1,
        temp_equation_file=False,
        delete_tempfiles=False,
        output_jax_format=False,
        output_torch_format=False,
        loss_scale="linear",
    )

    print("\nStart PySR symbolic regression training...")
    model.fit(X_train, y_train)

    equations = model.equations_
    if isinstance(equations, list):
        equations = equations[0]

    best_idx = select_best_equation_by_test_mae(
        model=model,
        equations=equations,
        X_test=X_test,
        y_test=y_test,
    )

    y_train_pred = model.predict(X_train, index=best_idx)
    y_test_pred = model.predict(X_test, index=best_idx)

    y_train_pred = np.nan_to_num(y_train_pred, nan=np.mean(y_train))
    y_test_pred = np.nan_to_num(y_test_pred, nan=np.mean(y_test))

    train_metrics = evaluate_prediction(y_train, y_train_pred)
    test_metrics = evaluate_prediction(y_test, y_test_pred)

    best_eq = equations.loc[best_idx]
    formula_str = best_eq["equation"]
    node_count = int(best_eq["complexity"])
    tree_depth = int(best_eq.get("depth", 0)) if "depth" in best_eq else 0
    r2_per_node = test_metrics["r2"] / node_count if node_count > 0 else 0.0

    print_evaluation_result(
        train_metrics=train_metrics,
        test_metrics=test_metrics,
        node_count=node_count,
        tree_depth=tree_depth,
        r2_per_node=r2_per_node,
        formula_str=formula_str,
    )

    save_evaluation_result(
        train_metrics=train_metrics,
        test_metrics=test_metrics,
        node_count=node_count,
        tree_depth=tree_depth,
        r2_per_node=r2_per_node,
        result_dir=RESULT_DIR,
    )

    sr_model = PySRWrapper(model, best_idx)

    return (
        sr_model,
        y_train_pred,
        y_test_pred,
        node_count,
        tree_depth,
        model,
        best_idx,
    )


def select_best_equation_by_test_mae(model, equations, X_test, y_test) -> int:
    """
    Select the equation with the lowest test-set MAE.
    """
    best_idx = equations.index[0]
    best_test_mae = float("inf")

    for idx, row in equations.iterrows():
        if pd.isna(row["equation"]):
            continue

        try:
            y_pred = model.predict(X_test, index=int(idx))
            test_mae = mean_absolute_error(y_test, y_pred)

            if test_mae < best_test_mae:
                best_test_mae = test_mae
                best_idx = int(idx)

        except Exception:
            continue

    return best_idx


class PySRWrapper:
    """
    A small wrapper to keep the selected PySR equation index.
    """
    def __init__(self, model, best_idx):
        self.model = model
        self.best_idx = best_idx

    @property
    def _program(self):
        return self.model.equations_.loc[self.best_idx]["equation"]

    def predict(self, X):
        return self.model.predict(X, index=self.best_idx)


# =============================================================================
# 5. Saving and reporting
# =============================================================================

def print_evaluation_result(
    train_metrics: dict,
    test_metrics: dict,
    node_count: int,
    tree_depth: int,
    r2_per_node: float,
    formula_str: str,
) -> None:
    """
    Print model performance and formula complexity.
    """
    print("\n=== Model evaluation ===")
    print(
        f"Train R²: {train_metrics['r2']:.4f} | "
        f"RMSE: {train_metrics['rmse']:.4f} | "
        f"MAE: {train_metrics['mae']:.4f} | "
        f"MAPE: {train_metrics['mape']:.2f}%"
    )
    print(
        f"Test  R²: {test_metrics['r2']:.4f} | "
        f"RMSE: {test_metrics['rmse']:.4f} | "
        f"MAE: {test_metrics['mae']:.4f} | "
        f"MAPE: {test_metrics['mape']:.2f}%"
    )

    print("\n=== Formula complexity ===")
    print(f"Formula: {formula_str}")
    print(f"Node count: {node_count}")
    print(f"Tree depth: {tree_depth}")
    print(f"R² per node: {r2_per_node:.4f}")


def save_evaluation_result(
    train_metrics: dict,
    test_metrics: dict,
    node_count: int,
    tree_depth: int,
    r2_per_node: float,
    result_dir: Path,
) -> None:
    """
    Save model evaluation metrics to CSV.
    """
    eval_result = pd.DataFrame({
        "metric": [
            "train_r2", "test_r2",
            "train_rmse", "test_rmse",
            "train_mae", "test_mae",
            "train_mape_percent", "test_mape_percent",
            "node_count", "tree_depth", "r2_per_node",
        ],
        "value": [
            train_metrics["r2"], test_metrics["r2"],
            train_metrics["rmse"], test_metrics["rmse"],
            train_metrics["mae"], test_metrics["mae"],
            train_metrics["mape"], test_metrics["mape"],
            node_count, tree_depth, r2_per_node,
        ],
    })

    eval_path = result_dir / "model_evaluation.csv"
    eval_result.to_csv(eval_path, index=False, encoding="utf-8")
    print(f"\nEvaluation results saved to: {eval_path}")


def visualize_result(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    sr_model,
    node_count: int,
    tree_depth: int,
) -> None:
    """
    Save a predicted-vs-true scatter plot and the selected symbolic formula.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    ax1.scatter(y_true, y_pred, alpha=0.6, s=20)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())

    ax1.plot(
        [min_val, max_val],
        [min_val, max_val],
        "r--",
        linewidth=2,
        label="Ideal line: y = x",
    )

    ax1.set_xlabel("True bilayer gray value")
    ax1.set_ylabel("Predicted bilayer gray value")
    ax1.set_title(f"Symbolic regression prediction\nR² = {r2_score(y_true, y_pred):.4f}")
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.axis("off")

    formula = str(sr_model._program)
    readable_formula = formula.replace("pow", "^").replace("/", "÷")

    text_content = (
        "bilayer_gray = f(layer1_gray, layer2_gray)\n\n"
        f"Best formula:\n{readable_formula}\n\n"
        "Formula complexity\n"
        f"Node count: {node_count}\n"
        f"Tree depth: {tree_depth}\n\n"
        "Variables\n"
        "x0: layer1_gray\n"
        "x1: layer2_gray"
    )

    ax2.text(
        0.05,
        0.5,
        text_content,
        fontsize=11,
        verticalalignment="center",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#f0f0f0", alpha=0.8),
    )

    img_path = RESULT_DIR / "symbolic_regression_result.png"

    plt.tight_layout()
    plt.savefig(img_path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Result figure saved to: {img_path}")


def save_formula_to_python(sr_model, node_count: int, tree_depth: int) -> None:
    """
    Save the selected symbolic formula as a standalone Python predictor.
    """
    formula_str = str(sr_model._program)

    expr = (
        formula_str
        .replace("x0", "layer1_gray")
        .replace("x1", "layer2_gray")
        .replace("pow", "np.power")
        .replace("^", "**")
    )

    code = f'''import numpy as np


def predict_bilayer_gray(layer1_gray, layer2_gray):
    """
    Predict bilayer gray value from layer1 and layer2 gray values.

    Formula source
    --------------
    PySR symbolic regression.

    Formula complexity
    ------------------
    Node count: {node_count}
    Tree depth: {tree_depth}

    Input
    -----
    layer1_gray, layer2_gray:
        Scalar, list, or NumPy array.

    Output
    ------
    Predicted bilayer gray value clipped to [0, 255].
    """
    layer1_gray = np.asarray(layer1_gray)
    layer2_gray = np.asarray(layer2_gray)

    bilayer_gray = {expr}
    bilayer_gray = np.clip(bilayer_gray, 0, 255)

    if bilayer_gray.ndim == 0:
        return float(bilayer_gray)

    return bilayer_gray


if __name__ == "__main__":
    print("Single-sample test")
    pred = predict_bilayer_gray(128, 135)
    print(f"layer1=128, layer2=135 -> bilayer={{pred:.2f}}")

    print("\\nBatch test")
    l1_arr = np.array([100, 120, 140, 160])
    l2_arr = np.array([110, 130, 150, 170])
    preds = predict_bilayer_gray(l1_arr, l2_arr)

    print(f"layer1: {{l1_arr}}")
    print(f"layer2: {{l2_arr}}")
    print(f"prediction: {{preds.round(2)}}")
'''

    py_path = RESULT_DIR / "bilayer_gray_predictor.py"

    with open(py_path, "w", encoding="utf-8") as f:
        f.write(code)

    print(f"Standalone predictor saved to: {py_path}")


def save_predictions_to_excel(
    train_metadata,
    X_train,
    y_train,
    y_train_pred,
    test_metadata,
    X_test,
    y_test,
    y_test_pred,
    result_dir: Path,
) -> None:
    """
    Save train/test predictions with source-image and pixel-index metadata.
    """
    train_df = pd.DataFrame({
        "source_image": train_metadata["source_image"],
        "pixel_index": train_metadata["pixel_index"],
        "layer1_gray": X_train[:, 0],
        "layer2_gray": X_train[:, 1],
        "bilayer_gray_true": y_train,
        "bilayer_gray_pred": y_train_pred,
    })

    test_df = pd.DataFrame({
        "source_image": test_metadata["source_image"],
        "pixel_index": test_metadata["pixel_index"],
        "layer1_gray": X_test[:, 0],
        "layer2_gray": X_test[:, 1],
        "bilayer_gray_true": y_test,
        "bilayer_gray_pred": y_test_pred,
    })

    train_metrics = evaluate_prediction(y_train, y_train_pred)
    test_metrics = evaluate_prediction(y_test, y_test_pred)

    summary_df = pd.DataFrame({
        "dataset": ["train", "test"],
        "sample_count": [len(train_df), len(test_df)],
        "r2": [train_metrics["r2"], test_metrics["r2"]],
        "rmse": [train_metrics["rmse"], test_metrics["rmse"]],
        "mae": [train_metrics["mae"], test_metrics["mae"]],
        "mape_percent": [train_metrics["mape"], test_metrics["mape"]],
        "mse": [train_metrics["mse"], test_metrics["mse"]],
    })

    excel_path = result_dir / "predictions_with_source.xlsx"

    with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
        train_df.to_excel(writer, sheet_name="train", index=False)
        test_df.to_excel(writer, sheet_name="test", index=False)
        summary_df.to_excel(writer, sheet_name="summary", index=False)

    print(f"Predictions with source metadata saved to: {excel_path}")


# =============================================================================
# 6. Main entry
# =============================================================================

if __name__ == "__main__":
    try:
        print("=== Start symbolic regression with PySR ===")

        gray_df = load_and_merge_gray_data(GRAY_DATA_DIR)

        feature_cols = ["layer1_gray", "layer2_gray"]
        target_col = "bilayer_gray"
        metadata_cols = ["source_image", "pixel_index"]

        train_df, test_df = train_test_split(
            gray_df,
            test_size=0.2,
            random_state=RANDOM_STATE,
            shuffle=True,
        )

        X_train = train_df[feature_cols].to_numpy(dtype=np.float64)
        y_train = train_df[target_col].to_numpy(dtype=np.float64)

        X_test = test_df[feature_cols].to_numpy(dtype=np.float64)
        y_test = test_df[target_col].to_numpy(dtype=np.float64)

        train_metadata = train_df[metadata_cols].reset_index(drop=True)
        test_metadata = test_df[metadata_cols].reset_index(drop=True)

        (
            sr_model,
            y_train_pred,
            y_test_pred,
            node_count,
            tree_depth,
            model,
            best_idx,
        ) = train_symbolic_regressor(
            X_train,
            X_test,
            y_train,
            y_test,
        )

        visualize_result(
            y_true=y_test,
            y_pred=y_test_pred,
            sr_model=sr_model,
            node_count=node_count,
            tree_depth=tree_depth,
        )

        save_formula_to_python(
            sr_model=sr_model,
            node_count=node_count,
            tree_depth=tree_depth,
        )

        save_predictions_to_excel(
            train_metadata=train_metadata,
            X_train=X_train,
            y_train=y_train,
            y_train_pred=y_train_pred,
            test_metadata=test_metadata,
            X_test=X_test,
            y_test=y_test,
            y_test_pred=y_test_pred,
            result_dir=RESULT_DIR,
        )

        raw_formula = str(sr_model._program)
        readable_formula = raw_formula.replace("pow", "^").replace("/", "÷")

        print("\n=== Final formula ===")
        print(f"Raw formula: {raw_formula}")
        print(f"Readable formula: {readable_formula}")
        print(f"\nAll results saved to: {RESULT_DIR}")

    except Exception as exc:
        print(f"Program failed: {exc}")

        import traceback
        traceback.print_exc()

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
=== Start symbolic regression with PySR ===
Loaded: gray_values_x-3.2_y-2.8_rot0_flip0.csv | samples: 400

Merged dataset size: 400 samples

Training samples: 320 | Test samples: 80

Start PySR symbolic regression training...


Compiling Julia backend...
[ Info: Started!



Expressions evaluated per second: 3.380e+04
Progress: 212 / 1000 total iterations (21.200%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Equation
1           2.625e+03  y = 113.35
3           1.769e+01  y = x₀ + x₁
5           1.442e-02  y = (x₀ + x₁) * 0.96731
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Press 'q' and then <enter> to stop execution early.

Expressions evaluated per second: 3.480e+04
Progress: 441 / 1000 total iterations (44.100%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity 

[ Info: Final population:
[ Info: Results saved to:


Result figure saved to: G:\DiffStack-code\Symbolic-regression\regression_results\computem4sym_absolute\symbolic_regression_result.png
Standalone predictor saved to: G:\DiffStack-code\Symbolic-regression\regression_results\computem4sym_absolute\bilayer_gray_predictor.py
Predictions with source metadata saved to: G:\DiffStack-code\Symbolic-regression\regression_results\computem4sym_absolute\predictions_with_source.xlsx

=== Final formula ===
Raw formula: (x0 + x1) * 0.9673063
Readable formula: (x0 + x1) * 0.9673063

All results saved to: G:\DiffStack-code\Symbolic-regression\regression_results\computem4sym_absolute
  - outputs\20260520_232257_AHKMKl\hall_of_fame.csv


# Trilayer

In [2]:
import csv
import random
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
from PIL import Image


# =============================================================================
# 1. Path and parameter settings
# =============================================================================

DATABASE_ROOT = Path(r"G:\DiffStack-code\Symbolic-regression\ReS2_trilayer")
OUTPUT_ROOT = Path(r"G:\DiffStack-code\Symbolic-regression\gray_results\ReS2_trilayer")

IMAGE_EXTENSIONS = (".png", ".tif", ".tiff")
TARGET_PIXEL_COUNT = 400
RANDOM_SEED = 21

# Your current directory structure should be:
# DATABASE_ROOT/layer1/data
# DATABASE_ROOT/layer2/data
# DATABASE_ROOT/layer3/data
# DATABASE_ROOT/trilayer/data
USE_NESTED_LAYER_DIR = True

# Filename example:
# 1.6_1.5_0.8_2.35_35.0_0.6_12345_type_trilayer.png
#
# The first four numbers are:
# move_x1, move_y1, move_x2, move_y2.
PARAM_PATTERN = re.compile(
    r"^([-+]?\d+(?:\.\d+)?)_"
    r"([-+]?\d+(?:\.\d+)?)_"
    r"([-+]?\d+(?:\.\d+)?)_"
    r"([-+]?\d+(?:\.\d+)?)_"
)

# Target trilayer stacking parameters in raw format:
# move_x1_move_y1_move_x2_move_y2
TARGET_STACK_PARAMS_LIST = [
    "1.6_1.5_0.8_2.35",
]


# =============================================================================
# 2. Basic utilities
# =============================================================================

def init_output_dir() -> None:
    """Create the output directory."""
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Output directory: {OUTPUT_ROOT}")


def extract_stack_params(filename: str) -> Optional[str]:
    """
    Extract the first four stacking parameters from a filename.

    Returns
    -------
    str or None
        Parameter key in the format:
        move_x1_move_y1_move_x2_move_y2
    """
    match = PARAM_PATTERN.match(filename)

    if match is None:
        return None

    v1, v2, v3, v4 = match.group(1), match.group(2), match.group(3), match.group(4)
    return f"{v1}_{v2}_{v3}_{v4}"


def get_layer_data_dir(layer_type: str) -> Path:
    """
    Return the image directory for a given layer.

    Supported directory layouts
    ---------------------------
    1. Nested layout:
       DATABASE_ROOT/layer1/data
       DATABASE_ROOT/layer2/data
       DATABASE_ROOT/layer3/data
       DATABASE_ROOT/trilayer/data

    2. Flat layout:
       DATABASE_ROOT/data_layer1
       DATABASE_ROOT/data_layer2
       DATABASE_ROOT/data_layer3
       DATABASE_ROOT/data_trilayer
    """
    if USE_NESTED_LAYER_DIR:
        return DATABASE_ROOT / layer_type / "data"

    return DATABASE_ROOT / f"data_{layer_type}"


def get_image_paths_by_layer(layer_type: str) -> Dict[str, Path]:
    """
    Build an image-path dictionary for a given layer.

    Returns
    -------
    dict
        Key   : stacking parameter key
        Value : image path
    """
    layer_dir = get_layer_data_dir(layer_type)
    image_paths = {}

    if not layer_dir.exists():
        print(f"Error: layer directory does not exist: {layer_dir}")
        return image_paths

    for file_path in layer_dir.iterdir():
        if not file_path.is_file():
            continue

        if file_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        params_key = extract_stack_params(file_path.name)

        if params_key is None:
            print(f"Skipped file with unrecognized parameter format: {file_path.name}")
            continue

        image_paths[params_key] = file_path

    print(f"{layer_type}: loaded {len(image_paths)} images")
    return image_paths


def load_image_as_gray_array(
    image_path: Path,
    target_size: Tuple[int, int] = (1024, 1024),
) -> Optional[np.ndarray]:
    """
    Load an image as a grayscale NumPy array.

    If the image size is not target_size, a center crop is applied.
    """
    try:
        with Image.open(image_path).convert("L") as img:
            if img.size != target_size:
                left = img.size[0] // 2 - target_size[0] // 2
                top = img.size[1] // 2 - target_size[1] // 2
                img = img.crop((left, top, left + target_size[0], top + target_size[1]))

            return np.asarray(img, dtype=np.float32)

    except Exception as exc:
        print(f"Failed to load image: {image_path.name}. Reason: {exc}")
        return None


# =============================================================================
# 3. Valid-pixel selection
# =============================================================================

def mean_filter_5x5(image_arr: np.ndarray) -> np.ndarray:
    """
    Compute the 5 x 5 local mean gray value for each pixel.

    Edge pixels are handled by edge padding.
    """
    height, width = image_arr.shape
    padded = np.pad(image_arr, pad_width=2, mode="edge")

    result = np.zeros_like(image_arr, dtype=np.float32)

    for dy in range(5):
        for dx in range(5):
            result += padded[dy:dy + height, dx:dx + width]

    return result / 25.0


def get_valid_pixels(
    layer1_arr: np.ndarray,
    layer2_arr: np.ndarray,
    layer3_arr: np.ndarray,
    trilayer_arr: np.ndarray,
    trilayer_threshold: float = 30.0,
) -> List[Tuple[int, int, float, float, float, float]]:
    """
    Select valid pixels for absolute gray-value relationship fitting.

    Current criterion
    -----------------
    The 5 x 5 local mean of the trilayer image must be larger than
    trilayer_threshold.

    Returns
    -------
    list
        Each item is:
        (x, y, layer1_gray, layer2_gray, layer3_gray, trilayer_gray)

        All gray values are absolute 5 x 5 local means.
        No centering or normalization is applied.
    """
    l1_avg = mean_filter_5x5(layer1_arr)
    l2_avg = mean_filter_5x5(layer2_arr)
    l3_avg = mean_filter_5x5(layer3_arr)
    tri_avg = mean_filter_5x5(trilayer_arr)

    mask = tri_avg > trilayer_threshold
    ys, xs = np.where(mask)

    return [
        (
            int(x),
            int(y),
            float(l1_avg[y, x]),
            float(l2_avg[y, x]),
            float(l3_avg[y, x]),
            float(tri_avg[y, x]),
        )
        for y, x in zip(ys, xs)
    ]


# =============================================================================
# 4. CSV writing and statistics
# =============================================================================

def compute_gray_stats(values: List[float]) -> Dict[str, float]:
    """Compute basic gray-value statistics."""
    arr = np.asarray(values, dtype=np.float32)

    mean = float(np.mean(arr))
    std = float(np.std(arr))
    cv = float(std / mean) if mean != 0 else 0.0
    q25, q75 = np.percentile(arr, [25, 75])

    return {
        "mean": mean,
        "std": std,
        "cv": cv,
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "q25": float(q25),
        "q75": float(q75),
    }


def write_pixel_csv(
    output_csv_path: Path,
    selected_pixels: List[Tuple[int, int, float, float, float, float]],
) -> None:
    """Save selected absolute gray values to CSV."""
    output_csv_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "x",
            "y",
            "layer1_gray",
            "layer2_gray",
            "layer3_gray",
            "trilayer_gray",
        ])
        writer.writerows(selected_pixels)


def append_summary_csv(
    summary_csv_path: Path,
    params_tag: str,
    selected_pixels: List[Tuple[int, int, float, float, float, float]],
) -> None:
    """Append gray-value statistics for one trilayer stacking configuration."""
    summary_csv_path.parent.mkdir(parents=True, exist_ok=True)

    l1_stats = compute_gray_stats([p[2] for p in selected_pixels])
    l2_stats = compute_gray_stats([p[3] for p in selected_pixels])
    l3_stats = compute_gray_stats([p[4] for p in selected_pixels])
    tri_stats = compute_gray_stats([p[5] for p in selected_pixels])

    file_exists = summary_csv_path.exists()

    header = [
        "params", "count",
        "l1_mean", "l1_std", "l1_cv", "l1_min", "l1_max", "l1_q25", "l1_q75",
        "l2_mean", "l2_std", "l2_cv", "l2_min", "l2_max", "l2_q25", "l2_q75",
        "l3_mean", "l3_std", "l3_cv", "l3_min", "l3_max", "l3_q25", "l3_q75",
        "tri_mean", "tri_std", "tri_cv", "tri_min", "tri_max", "tri_q25", "tri_q75",
    ]

    row = [
        params_tag, len(selected_pixels),
        l1_stats["mean"], l1_stats["std"], l1_stats["cv"], l1_stats["min"], l1_stats["max"], l1_stats["q25"], l1_stats["q75"],
        l2_stats["mean"], l2_stats["std"], l2_stats["cv"], l2_stats["min"], l2_stats["max"], l2_stats["q25"], l2_stats["q75"],
        l3_stats["mean"], l3_stats["std"], l3_stats["cv"], l3_stats["min"], l3_stats["max"], l3_stats["q25"], l3_stats["q75"],
        tri_stats["mean"], tri_stats["std"], tri_stats["cv"], tri_stats["min"], tri_stats["max"], tri_stats["q25"], tri_stats["q75"],
    ]

    with open(summary_csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        if not file_exists:
            writer.writerow(header)

        writer.writerow(row)


# =============================================================================
# 5. Single-stack extraction
# =============================================================================

def extract_gray_values_for_stack(
    layer1_path: Path,
    layer2_path: Path,
    layer3_path: Path,
    trilayer_path: Path,
    output_csv_path: Path,
    summary_csv_path: Optional[Path] = None,
) -> bool:
    """
    Extract absolute gray values from one layer1/layer2/layer3/trilayer image set.

    No centering or normalization is applied.
    """
    l1_arr = load_image_as_gray_array(layer1_path)
    l2_arr = load_image_as_gray_array(layer2_path)
    l3_arr = load_image_as_gray_array(layer3_path)
    tri_arr = load_image_as_gray_array(trilayer_path)

    if any(arr is None for arr in [l1_arr, l2_arr, l3_arr, tri_arr]):
        print("Skipped: at least one image failed to load")
        return False

    if not (l1_arr.shape == l2_arr.shape == l3_arr.shape == tri_arr.shape):
        print(
            "Skipped: image sizes are inconsistent "
            f"layer1={l1_arr.shape}, layer2={l2_arr.shape}, "
            f"layer3={l3_arr.shape}, trilayer={tri_arr.shape}"
        )
        return False

    valid_pixels = get_valid_pixels(
        layer1_arr=l1_arr,
        layer2_arr=l2_arr,
        layer3_arr=l3_arr,
        trilayer_arr=tri_arr,
        trilayer_threshold=30.0,
    )

    if not valid_pixels:
        print("Skipped: no valid pixels found")
        return False

    selected_pixels = (
        random.sample(valid_pixels, TARGET_PIXEL_COUNT)
        if len(valid_pixels) > TARGET_PIXEL_COUNT
        else valid_pixels
    )

    write_pixel_csv(output_csv_path, selected_pixels)

    if summary_csv_path is not None:
        params_tag = output_csv_path.stem.replace("gray_values_", "")
        append_summary_csv(summary_csv_path, params_tag, selected_pixels)

    print(f"Saved: {output_csv_path.name}, valid pixels: {len(selected_pixels)}")
    return True


# =============================================================================
# 6. Batch extraction
# =============================================================================

def batch_extract_gray_values() -> None:
    """Batch extract absolute gray values for TARGET_STACK_PARAMS_LIST."""
    init_output_dir()

    layer1_paths = get_image_paths_by_layer("layer1")
    layer2_paths = get_image_paths_by_layer("layer2")
    layer3_paths = get_image_paths_by_layer("layer3")
    trilayer_paths = get_image_paths_by_layer("trilayer")

    summary_csv_path = OUTPUT_ROOT / "summary_trilayer_gray_stats.csv"

    success_count = 0

    for raw_params in TARGET_STACK_PARAMS_LIST:
        print(f"\nProcessing: {raw_params}")

        missing_layers = [
            layer_name
            for layer_name, paths in [
                ("layer1", layer1_paths),
                ("layer2", layer2_paths),
                ("layer3", layer3_paths),
                ("trilayer", trilayer_paths),
            ]
            if raw_params not in paths
        ]

        if missing_layers:
            print(f"Skipped: missing {missing_layers}")
            continue

        output_csv_path = OUTPUT_ROOT / f"gray_values_{raw_params}.csv"

        ok = extract_gray_values_for_stack(
            layer1_path=layer1_paths[raw_params],
            layer2_path=layer2_paths[raw_params],
            layer3_path=layer3_paths[raw_params],
            trilayer_path=trilayer_paths[raw_params],
            output_csv_path=output_csv_path,
            summary_csv_path=summary_csv_path,
        )

        if ok:
            success_count += 1

    total_count = len(TARGET_STACK_PARAMS_LIST)

    print("\nBatch extraction finished")
    print(f"Requested groups: {total_count}")
    print(f"Successful groups: {success_count}")
    print(f"Failed or missing groups: {total_count - success_count}")
    print(f"Result directory: {OUTPUT_ROOT}")


# =============================================================================
# 7. Main entry
# =============================================================================

if __name__ == "__main__":
    random.seed(RANDOM_SEED)
    batch_extract_gray_values()

Output directory: G:\DiffStack-code\Symbolic-regression\gray_results\ReS2_trilayer
layer1: loaded 1 images
layer2: loaded 1 images
layer3: loaded 1 images
trilayer: loaded 1 images

Processing: 1.6_1.5_0.8_2.35
Saved: gray_values_1.6_1.5_0.8_2.35.csv, valid pixels: 400

Batch extraction finished
Requested groups: 1
Successful groups: 1
Failed or missing groups: 0
Result directory: G:\DiffStack-code\Symbolic-regression\gray_results\ReS2_trilayer


In [3]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pysr import PySRRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")


# =============================================================================
# 1. Path and training settings
# =============================================================================

GRAY_DATA_DIR = Path(r"G:\DiffStack-code\Symbolic-regression\gray_results\ReS2_trilayer")
RESULT_DIR = Path(r"G:\DiffStack-code\Symbolic-regression\regression_results\ReS2_trilayer_absolute")

# POPULATION_SIZE = 1000
GENERATIONS = 100
PARSIMONY_COEFFICIENT = 0.5
RANDOM_STATE = 42

RESULT_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 2. Data loading
# =============================================================================

def load_and_merge_gray_data(data_dir: Path) -> pd.DataFrame:
    """
    Load all gray-value CSV files and merge them into one DataFrame.

    Expected CSV columns
    --------------------
    layer1_gray
    layer2_gray
    layer3_gray
    trilayer_gray

    Additional metadata columns are added:
    source_image
    pixel_index
    """
    all_data = []
    required_cols = ["layer1_gray", "layer2_gray", "layer3_gray", "trilayer_gray"]

    for csv_path in sorted(data_dir.glob("gray_values_*.csv")):
        df = pd.read_csv(csv_path)

        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            print(f"Skipped {csv_path.name}: missing columns {missing_cols}")
            continue

        df = df[required_cols].copy()
        df["source_image"] = csv_path.name
        df["pixel_index"] = df.index

        all_data.append(df)
        print(f"Loaded: {csv_path.name} | samples: {len(df)}")

    if not all_data:
        raise ValueError(f"No valid gray-value CSV files found in: {data_dir}")

    merged_df = pd.concat(all_data, ignore_index=True)
    print(f"\nMerged dataset size: {len(merged_df)} samples")

    return merged_df


# =============================================================================
# 3. Metrics
# =============================================================================

def mean_absolute_percentage_error(y_true, y_pred) -> float:
    """
    Compute MAPE while ignoring zero-valued targets.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mask = y_true != 0
    if not np.any(mask):
        return np.nan

    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def evaluate_prediction(y_true, y_pred) -> dict:
    """
    Compute regression metrics.
    """
    return {
        "r2": r2_score(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mae": mean_absolute_error(y_true, y_pred),
        "mape": mean_absolute_percentage_error(y_true, y_pred),
        "mse": mean_squared_error(y_true, y_pred),
    }


# =============================================================================
# 4. Symbolic regression
# =============================================================================

def train_symbolic_regressor(X_train, X_test, y_train, y_test):
    """
    Train a PySR symbolic regression model.

    Model target
    ------------
    trilayer_gray = f(layer1_gray, layer2_gray, layer3_gray)

    The input gray values are absolute gray values.
    No centering or normalization is applied.
    """
    print(f"\nTraining samples: {len(X_train)} | Test samples: {len(X_test)}")

    model = PySRRegressor(
        niterations=GENERATIONS,
        # populations=POPULATION_SIZE,
        binary_operators=["+", "-", "*", "/", "pow"],
        unary_operators=[],
        maxsize=10,
        parsimony=PARSIMONY_COEFFICIENT,
        elementwise_loss="L2DistLoss()",
        model_selection="best",
        random_state=RANDOM_STATE,
        verbosity=1,
        temp_equation_file=False,
        delete_tempfiles=False,
        output_jax_format=False,
        output_torch_format=False,
        loss_scale="linear",
    )

    print("\nStart PySR symbolic regression training...")
    model.fit(X_train, y_train)

    equations = model.equations_
    if isinstance(equations, list):
        equations = equations[0]

    best_idx = select_best_equation_by_test_mae(
        model=model,
        equations=equations,
        X_test=X_test,
        y_test=y_test,
    )

    y_train_pred = model.predict(X_train, index=best_idx)
    y_test_pred = model.predict(X_test, index=best_idx)

    y_train_pred = np.nan_to_num(y_train_pred, nan=np.mean(y_train))
    y_test_pred = np.nan_to_num(y_test_pred, nan=np.mean(y_test))

    train_metrics = evaluate_prediction(y_train, y_train_pred)
    test_metrics = evaluate_prediction(y_test, y_test_pred)

    best_eq = equations.loc[best_idx]
    formula_str = best_eq["equation"]
    node_count = int(best_eq["complexity"])
    tree_depth = int(best_eq.get("depth", 0)) if "depth" in best_eq else 0
    r2_per_node = test_metrics["r2"] / node_count if node_count > 0 else 0.0

    print_evaluation_result(
        train_metrics=train_metrics,
        test_metrics=test_metrics,
        node_count=node_count,
        tree_depth=tree_depth,
        r2_per_node=r2_per_node,
        formula_str=formula_str,
    )

    save_evaluation_result(
        train_metrics=train_metrics,
        test_metrics=test_metrics,
        node_count=node_count,
        tree_depth=tree_depth,
        r2_per_node=r2_per_node,
        result_dir=RESULT_DIR,
    )

    sr_model = PySRWrapper(model, best_idx)

    return sr_model, y_train_pred, y_test_pred, node_count, tree_depth, model, best_idx


def select_best_equation_by_test_mae(model, equations, X_test, y_test) -> int:
    """
    Select the equation with the lowest test-set MAE.
    """
    best_idx = equations.index[0]
    best_test_mae = float("inf")

    for idx, row in equations.iterrows():
        if pd.isna(row["equation"]):
            continue

        try:
            y_pred = model.predict(X_test, index=int(idx))
            test_mae = mean_absolute_error(y_test, y_pred)

            if test_mae < best_test_mae:
                best_test_mae = test_mae
                best_idx = int(idx)

        except Exception:
            continue

    return best_idx


class PySRWrapper:
    """
    A small wrapper to keep the selected PySR equation index.
    """

    def __init__(self, model, best_idx):
        self.model = model
        self.best_idx = best_idx

    @property
    def _program(self):
        return self.model.equations_.loc[self.best_idx]["equation"]

    def predict(self, X):
        return self.model.predict(X, index=self.best_idx)


# =============================================================================
# 5. Reporting and saving
# =============================================================================

def print_evaluation_result(
    train_metrics: dict,
    test_metrics: dict,
    node_count: int,
    tree_depth: int,
    r2_per_node: float,
    formula_str: str,
) -> None:
    """
    Print model performance and formula complexity.
    """
    print("\n=== Model evaluation ===")
    print(
        f"Train R²: {train_metrics['r2']:.4f} | "
        f"RMSE: {train_metrics['rmse']:.4f} | "
        f"MAE: {train_metrics['mae']:.4f} | "
        f"MAPE: {train_metrics['mape']:.2f}%"
    )
    print(
        f"Test  R²: {test_metrics['r2']:.4f} | "
        f"RMSE: {test_metrics['rmse']:.4f} | "
        f"MAE: {test_metrics['mae']:.4f} | "
        f"MAPE: {test_metrics['mape']:.2f}%"
    )

    print("\n=== Formula complexity ===")
    print(f"Formula: {formula_str}")
    print(f"Node count: {node_count}")
    print(f"Tree depth: {tree_depth}")
    print(f"R² per node: {r2_per_node:.4f}")


def save_evaluation_result(
    train_metrics: dict,
    test_metrics: dict,
    node_count: int,
    tree_depth: int,
    r2_per_node: float,
    result_dir: Path,
) -> None:
    """
    Save model evaluation metrics to CSV.
    """
    eval_result = pd.DataFrame({
        "metric": [
            "train_r2", "test_r2",
            "train_rmse", "test_rmse",
            "train_mae", "test_mae",
            "train_mape_percent", "test_mape_percent",
            "node_count", "tree_depth", "r2_per_node",
        ],
        "value": [
            train_metrics["r2"], test_metrics["r2"],
            train_metrics["rmse"], test_metrics["rmse"],
            train_metrics["mae"], test_metrics["mae"],
            train_metrics["mape"], test_metrics["mape"],
            node_count, tree_depth, r2_per_node,
        ],
    })

    eval_path = result_dir / "model_evaluation.csv"
    eval_result.to_csv(eval_path, index=False, encoding="utf-8")
    print(f"\nEvaluation results saved to: {eval_path}")


def visualize_result(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    sr_model,
    node_count: int,
    tree_depth: int,
) -> None:
    """
    Save a predicted-vs-true scatter plot and the selected symbolic formula.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    ax1.scatter(y_true, y_pred, alpha=0.6, s=20)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())

    ax1.plot(
        [min_val, max_val],
        [min_val, max_val],
        "r--",
        linewidth=2,
        label="Ideal line: y = x",
    )

    ax1.set_xlabel("True trilayer gray value")
    ax1.set_ylabel("Predicted trilayer gray value")
    ax1.set_title(f"Symbolic regression prediction\nR² = {r2_score(y_true, y_pred):.4f}")
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.axis("off")

    formula = str(sr_model._program)
    readable_formula = formula.replace("pow", "^").replace("/", "÷")

    text_content = (
        "trilayer_gray = f(layer1_gray, layer2_gray, layer3_gray)\n\n"
        f"Best formula:\n{readable_formula}\n\n"
        "Formula complexity\n"
        f"Node count: {node_count}\n"
        f"Tree depth: {tree_depth}\n\n"
        "Variables\n"
        "x0: layer1_gray\n"
        "x1: layer2_gray\n"
        "x2: layer3_gray"
    )

    ax2.text(
        0.05,
        0.5,
        text_content,
        fontsize=11,
        verticalalignment="center",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#f0f0f0", alpha=0.8),
    )

    img_path = RESULT_DIR / "symbolic_regression_result.png"

    plt.tight_layout()
    plt.savefig(img_path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Result figure saved to: {img_path}")


def save_formula_to_python(sr_model, node_count: int, tree_depth: int) -> None:
    """
    Save the selected symbolic formula as a standalone Python predictor.
    """
    formula_str = str(sr_model._program)

    expr = (
        formula_str
        .replace("x0", "layer1_gray")
        .replace("x1", "layer2_gray")
        .replace("x2", "layer3_gray")
        .replace("pow", "np.power")
        .replace("^", "**")
    )

    code = f'''import numpy as np


def predict_trilayer_gray(layer1_gray, layer2_gray, layer3_gray):
    """
    Predict trilayer gray value from three monolayer gray values.

    Formula source
    --------------
    PySR symbolic regression.

    Formula complexity
    ------------------
    Node count: {node_count}
    Tree depth: {tree_depth}

    Input
    -----
    layer1_gray, layer2_gray, layer3_gray:
        Scalar, list, or NumPy array.

    Output
    ------
    Predicted trilayer gray value clipped to [0, 255].
    """
    layer1_gray = np.asarray(layer1_gray)
    layer2_gray = np.asarray(layer2_gray)
    layer3_gray = np.asarray(layer3_gray)

    trilayer_gray = {expr}
    trilayer_gray = np.clip(trilayer_gray, 0, 255)

    if trilayer_gray.ndim == 0:
        return float(trilayer_gray)

    return trilayer_gray


if __name__ == "__main__":
    print("Single-sample test")
    pred = predict_trilayer_gray(128, 135, 140)
    print(f"layer1=128, layer2=135, layer3=140 -> trilayer={{pred:.2f}}")

    print("\\nBatch test")
    l1_arr = np.array([100, 120, 140, 160])
    l2_arr = np.array([110, 130, 150, 170])
    l3_arr = np.array([105, 125, 145, 165])

    preds = predict_trilayer_gray(l1_arr, l2_arr, l3_arr)

    print(f"layer1: {{l1_arr}}")
    print(f"layer2: {{l2_arr}}")
    print(f"layer3: {{l3_arr}}")
    print(f"prediction: {{preds.round(2)}}")
'''

    py_path = RESULT_DIR / "trilayer_gray_predictor.py"

    with open(py_path, "w", encoding="utf-8") as f:
        f.write(code)

    print(f"Standalone predictor saved to: {py_path}")


def save_predictions_to_excel(
    train_metadata,
    X_train,
    y_train,
    y_train_pred,
    test_metadata,
    X_test,
    y_test,
    y_test_pred,
    result_dir: Path,
) -> None:
    """
    Save train/test predictions with source-image and pixel-index metadata.
    """
    train_df = pd.DataFrame({
        "source_image": train_metadata["source_image"],
        "pixel_index": train_metadata["pixel_index"],
        "layer1_gray": X_train[:, 0],
        "layer2_gray": X_train[:, 1],
        "layer3_gray": X_train[:, 2],
        "trilayer_gray_true": y_train,
        "trilayer_gray_pred": y_train_pred,
    })

    test_df = pd.DataFrame({
        "source_image": test_metadata["source_image"],
        "pixel_index": test_metadata["pixel_index"],
        "layer1_gray": X_test[:, 0],
        "layer2_gray": X_test[:, 1],
        "layer3_gray": X_test[:, 2],
        "trilayer_gray_true": y_test,
        "trilayer_gray_pred": y_test_pred,
    })

    train_metrics = evaluate_prediction(y_train, y_train_pred)
    test_metrics = evaluate_prediction(y_test, y_test_pred)

    summary_df = pd.DataFrame({
        "dataset": ["train", "test"],
        "sample_count": [len(train_df), len(test_df)],
        "r2": [train_metrics["r2"], test_metrics["r2"]],
        "rmse": [train_metrics["rmse"], test_metrics["rmse"]],
        "mae": [train_metrics["mae"], test_metrics["mae"]],
        "mape_percent": [train_metrics["mape"], test_metrics["mape"]],
        "mse": [train_metrics["mse"], test_metrics["mse"]],
    })

    excel_path = result_dir / "predictions_with_source.xlsx"

    with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
        train_df.to_excel(writer, sheet_name="train", index=False)
        test_df.to_excel(writer, sheet_name="test", index=False)
        summary_df.to_excel(writer, sheet_name="summary", index=False)

    print(f"Predictions with source metadata saved to: {excel_path}")


# =============================================================================
# 6. Main entry
# =============================================================================

if __name__ == "__main__":
    try:
        print("=== Start trilayer symbolic regression with PySR ===")

        gray_df = load_and_merge_gray_data(GRAY_DATA_DIR)

        feature_cols = ["layer1_gray", "layer2_gray", "layer3_gray"]
        target_col = "trilayer_gray"
        metadata_cols = ["source_image", "pixel_index"]

        train_df, test_df = train_test_split(
            gray_df,
            test_size=0.2,
            random_state=RANDOM_STATE,
            shuffle=True,
        )

        X_train = train_df[feature_cols].to_numpy(dtype=np.float64)
        y_train = train_df[target_col].to_numpy(dtype=np.float64)

        X_test = test_df[feature_cols].to_numpy(dtype=np.float64)
        y_test = test_df[target_col].to_numpy(dtype=np.float64)

        train_metadata = train_df[metadata_cols].reset_index(drop=True)
        test_metadata = test_df[metadata_cols].reset_index(drop=True)

        (
            sr_model,
            y_train_pred,
            y_test_pred,
            node_count,
            tree_depth,
            model,
            best_idx,
        ) = train_symbolic_regressor(
            X_train,
            X_test,
            y_train,
            y_test,
        )

        visualize_result(
            y_true=y_test,
            y_pred=y_test_pred,
            sr_model=sr_model,
            node_count=node_count,
            tree_depth=tree_depth,
        )

        save_formula_to_python(
            sr_model=sr_model,
            node_count=node_count,
            tree_depth=tree_depth,
        )

        save_predictions_to_excel(
            train_metadata=train_metadata,
            X_train=X_train,
            y_train=y_train,
            y_train_pred=y_train_pred,
            test_metadata=test_metadata,
            X_test=X_test,
            y_test=y_test,
            y_test_pred=y_test_pred,
            result_dir=RESULT_DIR,
        )

        raw_formula = str(sr_model._program)
        readable_formula = raw_formula.replace("pow", "^").replace("/", "÷")

        print("\n=== Final formula ===")
        print(f"Raw formula: {raw_formula}")
        print(f"Readable formula: {readable_formula}")
        print(f"\nAll results saved to: {RESULT_DIR}")

    except Exception as exc:
        print(f"Program failed: {exc}")

        import traceback
        traceback.print_exc()

=== Start trilayer symbolic regression with PySR ===
Loaded: gray_values_1.6_1.5_0.8_2.35.csv | samples: 400

Merged dataset size: 400 samples

Training samples: 320 | Test samples: 80

Start PySR symbolic regression training...


[ Info: Started!



Expressions evaluated per second: 5.330e+04
Progress: 332 / 3100 total iterations (10.710%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Equation
1           2.131e+03  y = 84.536
3           1.556e+03  y = x₀ - -31.022
5           9.590e+02  y = (x₀ - -91.367) * 0.58352
7           1.019e-02  y = (x₂ + (x₀ + x₁)) * 0.54773
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Press 'q' and then <enter> to stop execution early.

Expressions evaluated per second: 5.700e+04
Progress: 709 / 3100 total iterations (22.871%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────

[ Info: Final population:
[ Info: Results saved to:


Result figure saved to: G:\DiffStack-code\Symbolic-regression\regression_results\ReS2_trilayer_absolute\symbolic_regression_result.png
Standalone predictor saved to: G:\DiffStack-code\Symbolic-regression\regression_results\ReS2_trilayer_absolute\trilayer_gray_predictor.py
Predictions with source metadata saved to: G:\DiffStack-code\Symbolic-regression\regression_results\ReS2_trilayer_absolute\predictions_with_source.xlsx

=== Final formula ===
Raw formula: (x2 + (x0 + x1)) * 0.5477291
Readable formula: (x2 + (x0 + x1)) * 0.5477291

All results saved to: G:\DiffStack-code\Symbolic-regression\regression_results\ReS2_trilayer_absolute
  - outputs\20260520_233425_nrJqpr\hall_of_fame.csv
